In [1]:
import pandas as pd

X_raw_train = pd.read_csv('../data/processed/X_raw_train.csv', index_col=0)
X_raw_test  = pd.read_csv('../data/processed/X_raw_test.csv',  index_col=0)
X_ext_train = pd.read_csv('../data/processed/X_ext_train.csv', index_col=0)
X_ext_test  = pd.read_csv('../data/processed/X_ext_test.csv',  index_col=0)
y_train     = pd.read_csv('../data/processed/y_train.csv',     index_col=0).squeeze()
y_test      = pd.read_csv('../data/processed/y_test.csv',      index_col=0).squeeze()

print("raw:", X_raw_train.shape, X_raw_test.shape)
print("ext:", X_ext_train.shape, X_ext_test.shape)
print("y:  ", y_train.shape, y_test.shape)
print(y_train.value_counts(normalize=True).round(3))

raw: (9668, 14) (2417, 14)
ext: (9668, 19) (2417, 19)
y:   (9668,) (2417,)
Disease_encoded
4    0.236
0    0.176
5    0.153
2    0.147
1    0.133
6    0.112
3    0.043
Name: proportion, dtype: float64


In [2]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

disease_names = ['Ankylosing Spondylitis', 'Normal', 'Psoriatic Arthritis',
                  'Reactive Arthritis', 'Rheumatoid Arthritis',
                  "Sjögren's Syndrome", 'Systemic Lupus Erythematosus']

def evaluate_model(y_true, y_pred, model_name):
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    weighted_f1 = f1_score(y_true, y_pred, average='weighted')

    print(f"=== {model_name} ===")
    print(f"Macro F1:    {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=disease_names, digits=3))

    cm = confusion_matrix(y_true, y_pred)
    return {'model': model_name, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
            'y_true': y_true, 'y_pred': y_pred, 'confusion_matrix': cm}

In [3]:


# multi_class handling: with 7 disease classes, LogisticRegression needs to decide
# how to extend binary logistic regression to multiclass. 'multinomial' fits one
# unified model that directly estimates probabilities across all 7 classes at once
# (more statistically correct than the older one-vs-rest default).
# class_weight='balanced' matters a lot here: it upweights the loss contribution of
# rare classes (Reactive Arthritis, 4.3% of data) so the model doesn't just learn to
# ignore them to minimize overall error — same spirit as why we use macro-F1 for scoring.
# max_iter raised from the sklearn default (100) because multinomial logistic regression
# on 9,668 rows with 14 features often needs more iterations to converge.
from sklearn.linear_model import LogisticRegression

# solver='lbfgs' (the default) always fits a genuine multinomial logistic regression
# for multiclass problems in scikit-learn 1.5+ — the old multi_class parameter was
# removed because there's no longer an explicit choice to make with this solver.
# (liblinear would force one-vs-rest, but we don't use that solver here.)
lr_raw = LogisticRegression(solver='lbfgs', class_weight='balanced', max_iter=1000, random_state=42)
lr_raw.fit(X_raw_train, y_train)

y_pred_lr_raw = lr_raw.predict(X_raw_test)
results_lr_raw = evaluate_model(y_test, y_pred_lr_raw, "Logistic Regression (raw features)")

=== Logistic Regression (raw features) ===
Macro F1:    0.8042
Weighted F1: 0.8107

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.642     0.569     0.603       425
                      Normal      0.887     0.860     0.873       321
         Psoriatic Arthritis      0.806     0.849     0.827       357
          Reactive Arthritis      0.485     0.932     0.638       103
        Rheumatoid Arthritis      0.869     0.777     0.820       570
          Sjögren's Syndrome      0.885     0.897     0.891       370
Systemic Lupus Erythematosus      0.978     0.974     0.976       271

                    accuracy                          0.809      2417
                   macro avg      0.793     0.837     0.804      2417
                weighted avg      0.820     0.809     0.811      2417



In [4]:
lr_ext = LogisticRegression(solver='lbfgs', class_weight='balanced', max_iter=1000, random_state=42)
lr_ext.fit(X_ext_train, y_train)

y_pred_lr_ext = lr_ext.predict(X_ext_test)
results_lr_ext = evaluate_model(y_test, y_pred_lr_ext, "Logistic Regression (extended features)")

=== Logistic Regression (extended features) ===
Macro F1:    0.8021
Weighted F1: 0.8090

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.635     0.565     0.598       425
                      Normal      0.876     0.860     0.868       321
         Psoriatic Arthritis      0.816     0.846     0.831       357
          Reactive Arthritis      0.475     0.932     0.630       103
        Rheumatoid Arthritis      0.863     0.775     0.817       570
          Sjögren's Syndrome      0.890     0.895     0.892       370
Systemic Lupus Erythematosus      0.985     0.974     0.980       271

                    accuracy                          0.807      2417
                   macro avg      0.792     0.835     0.802      2417
                weighted avg      0.819     0.807     0.809      2417



In [5]:
from sklearn.ensemble import RandomForestClassifier

# n_estimators: number of trees in the forest. More trees = more stable averaging,
# with diminishing returns past a few hundred. 300 is a reasonable, unremarkable baseline —
# no tuning yet, that comes in a later stage.
# max_depth left at default (None = trees grow until leaves are pure or too small to split).
# This sounds reckless, but bagging is exactly what protects against the overfitting
# this would normally cause in a single tree.
# n_jobs=-1: use all available CPU cores to train trees in parallel (they're independent
# of each other, so this is free parallelism, not an approximation).
rf_raw = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
rf_raw.fit(X_raw_train, y_train)

y_pred_rf_raw = rf_raw.predict(X_raw_test)
results_rf_raw = evaluate_model(y_test, y_pred_rf_raw, "Random Forest (raw features)")

=== Random Forest (raw features) ===
Macro F1:    0.8370
Weighted F1: 0.8399

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.780     0.560     0.652       425
                      Normal      0.889     0.850     0.869       321
         Psoriatic Arthritis      0.829     0.910     0.868       357
          Reactive Arthritis      0.656     0.816     0.727       103
        Rheumatoid Arthritis      0.820     0.905     0.861       570
          Sjögren's Syndrome      0.870     0.922     0.895       370
Systemic Lupus Erythematosus      1.000     0.974     0.987       271

                    accuracy                          0.844      2417
                   macro avg      0.835     0.848     0.837      2417
                weighted avg      0.844     0.844     0.840      2417



In [6]:
import pandas as pd

importances = pd.Series(rf_raw.feature_importances_, index=X_raw_train.columns).sort_values(ascending=False)
print(importances)

ESR           0.220546
CRP           0.139766
RF            0.102804
C3            0.100613
Anti-CCP      0.096205
C4            0.085577
HLA-B27       0.050200
ANA           0.045000
Anti-La       0.035275
Anti-Ro       0.033916
Anti-Sm       0.030306
Anti-dsDNA    0.028217
Age           0.026321
Gender        0.005256
dtype: float64


In [7]:
rf_ext = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
rf_ext.fit(X_ext_train, y_train)

y_pred_rf_ext = rf_ext.predict(X_ext_test)
results_rf_ext = evaluate_model(y_test, y_pred_rf_ext, "Random Forest (extended features)")

=== Random Forest (extended features) ===
Macro F1:    0.8344
Weighted F1: 0.8375

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.757     0.551     0.638       425
                      Normal      0.899     0.857     0.877       321
         Psoriatic Arthritis      0.827     0.913     0.868       357
          Reactive Arthritis      0.636     0.816     0.715       103
        Rheumatoid Arthritis      0.821     0.893     0.855       570
          Sjögren's Syndrome      0.875     0.927     0.900       370
Systemic Lupus Erythematosus      1.000     0.974     0.987       271

                    accuracy                          0.842      2417
                   macro avg      0.831     0.847     0.834      2417
                weighted avg      0.842     0.842     0.838      2417



In [8]:
import os

os.makedirs('../results', exist_ok=True)

def log_result(results_dict):
    return {
        'model': results_dict['model'],
        'macro_f1': round(results_dict['macro_f1'], 4),
        'weighted_f1': round(results_dict['weighted_f1'], 4),
    }

stage1_log = pd.DataFrame([
    log_result(results_lr_raw),
    log_result(results_lr_ext),
    log_result(results_rf_raw),
    log_result(results_rf_ext),
])

stage1_log.to_csv('../results/stage1_summary.csv', index=False)
stage1_log

,model,macro_f1,weighted_f1
0,Logistic Regression (raw features),0.8042,0.8107
1,Logistic Regression (extended features),0.8021,0.8090
2,Random Forest (raw features),0.8370,0.8399
3,Random Forest (extended features),0.8344,0.8375


In [9]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [10]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

xgb_raw = XGBClassifier(
    n_estimators=300,      # boosting rounds — same scale as RF's tree count, different role
    max_depth=5,           # shallow on purpose — see table above
    learning_rate=0.1,     # shrinkage: how much each tree's correction counts (lower = more conservative, needs more rounds)
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
xgb_raw.fit(X_raw_train, y_train, sample_weight=sample_weights)

y_pred_xgb_raw = xgb_raw.predict(X_raw_test)
results_xgb_raw = evaluate_model(y_test, y_pred_xgb_raw, "XGBoost (raw features)")

=== XGBoost (raw features) ===
Macro F1:    0.8346
Weighted F1: 0.8369

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.707     0.624     0.662       425
                      Normal      0.874     0.863     0.868       321
         Psoriatic Arthritis      0.848     0.905     0.875       357
          Reactive Arthritis      0.646     0.796     0.713       103
        Rheumatoid Arthritis      0.834     0.840     0.837       570
          Sjögren's Syndrome      0.886     0.900     0.893       370
Systemic Lupus Erythematosus      1.000     0.985     0.993       271

                    accuracy                          0.838      2417
                   macro avg      0.828     0.845     0.835      2417
                weighted avg      0.838     0.838     0.837      2417



In [11]:
stage1_log = pd.concat([stage1_log, pd.DataFrame([log_result(results_xgb_raw)])], ignore_index=True)
stage1_log.to_csv('../results/stage1_summary.csv', index=False)
stage1_log

,model,macro_f1,weighted_f1
0,Logistic Regression (raw features),0.8042,0.8107
1,Logistic Regression (extended features),0.8021,0.8090
2,Random Forest (raw features),0.8370,0.8399
3,Random Forest (extended features),0.8344,0.8375
4,XGBoost (raw features),0.8346,0.8369


In [12]:
xgb_ext = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                         objective='multi:softprob', eval_metric='mlogloss',
                         random_state=42, n_jobs=-1)
xgb_ext.fit(X_ext_train, y_train, sample_weight=sample_weights)

y_pred_xgb_ext = xgb_ext.predict(X_ext_test)
results_xgb_ext = evaluate_model(y_test, y_pred_xgb_ext, "XGBoost (extended features)")

=== XGBoost (extended features) ===
Macro F1:    0.8347
Weighted F1: 0.8390

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.723     0.619     0.667       425
                      Normal      0.860     0.863     0.862       321
         Psoriatic Arthritis      0.850     0.905     0.877       357
          Reactive Arthritis      0.648     0.786     0.711       103
        Rheumatoid Arthritis      0.842     0.868     0.855       570
          Sjögren's Syndrome      0.879     0.881     0.880       370
Systemic Lupus Erythematosus      1.000     0.985     0.993       271

                    accuracy                          0.841      2417
                   macro avg      0.829     0.844     0.835      2417
                weighted avg      0.840     0.841     0.839      2417



In [13]:
stage1_log = pd.concat([stage1_log, pd.DataFrame([log_result(results_xgb_ext)])], ignore_index=True)
stage1_log.to_csv('../results/stage1_summary.csv', index=False)
stage1_log

,model,macro_f1,weighted_f1
0,Logistic Regression (raw features),0.8042,0.8107
1,Logistic Regression (extended features),0.8021,0.8090
2,Random Forest (raw features),0.8370,0.8399
3,Random Forest (extended features),0.8344,0.8375
4,XGBoost (raw features),0.8346,0.8369
5,XGBoost (extended features),0.8347,0.8390
